# 01 — Data extraction and validation

This notebook loads the United States GDP data from 1920 onward. The default historical source is Maddison Project Database 2023. FRED/BEA annual real GDP is used for validation from 1929 onward.

In [ ]:
from pathlib import Path
import pandas as pd

from us_gdp_regime.config import load_config
from us_gdp_regime.pipeline import _load_primary_series, _maybe_build_fred_validation
from us_gdp_regime.transform import validate_gdp_series

config = load_config(Path('..') / 'config' / 'default.yaml')


## Load primary source

The project uses Maddison because the target window begins in 1920. FRED/BEA annual real GDP starts later, in 1929, so it is better used as a validation source for the overlap.

In [ ]:
series = validate_gdp_series(_load_primary_series(config))
series.head(), series.tail()


In [ ]:
series.describe(include='all')


## Missing values and growth-rate checks

In [ ]:
series.isna().sum()


In [ ]:
ax = series.plot(x='year', y='gdp_growth', figsize=(12, 5), marker='o', title='United States real GDP growth')
ax.set_xlabel('Year')
ax.set_ylabel('Annual growth, %')


## Optional FRED/BEA validation

In [ ]:
comparison = _maybe_build_fred_validation(config, series)
if comparison is not None:
    display(comparison.head())
    display(comparison[['growth_maddison', 'growth_fred', 'growth_difference']].describe())
else:
    print('FRED validation was not available. Run with network access or place the FRED CSV in data/raw/.')


In [ ]:
output = Path('..') / 'data' / 'processed' / 'us_gdp_series.csv'
output.parent.mkdir(parents=True, exist_ok=True)
series.to_csv(output, index=False)
output
